# Notebook 7 — Round Outcomes & Fire Power Prediction

**Dataset:** 741 battles, 51 bots, ~48M tick rows

**Questions:**
5. Can we predict opponent fire POWER (not just timing)?
6. What determines round outcomes?
10. Do the 3 game-state clusters hold at scale?
11. Are there "winning" game states?

**Key concepts:**
- **Fire power** ranges from 0.1 to 3.0. Higher power = more damage but slower bullet = easier to dodge.
  Smart bots adjust power based on distance (close → high power, far → low power) and energy.
- **Round outcome** = which bot has more energy / deals more damage. We have `scores.csv` per round.
- **Game state clusters** — the initial analysis found 3 clusters (close/mid/long range). With 50 bots,
  we may find more nuanced states.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
CSV_ROOT = Path('../output/csv')

all_scores_files = sorted(CSV_ROOT.rglob('scores.csv'))
all_ticks_files = sorted(CSV_ROOT.rglob('ticks.csv'))
all_waves_files = sorted(CSV_ROOT.rglob('waves.csv'))
print(f'Scores files: {len(all_scores_files)}')
print(f'Ticks files: {len(all_ticks_files)}')
print(f'Waves files: {len(all_waves_files)}')

## 1. Load Scores Data

scores.csv is small — one row per round per perspective. We can load ALL of them.

In [ ]:
# Load all scores
score_frames = []
for f in all_scores_files:
    bot_name = f.parent.name
    battle_dir = f.parent.parent
    perspectives = [d.name for d in battle_dir.iterdir() if d.is_dir()]
    opponent_name = [p for p in perspectives if p != bot_name]
    if not opponent_name:
        continue
    try:
        df = pd.read_csv(f)
        df['bot'] = bot_name
        df['opponent'] = opponent_name[0]
        score_frames.append(df)
    except Exception:
        pass

scores = pd.concat(score_frames, ignore_index=True)
print(f'Total score rows: {len(scores):,}')
print(f'Unique bots: {scores["bot"].nunique()}')
print(f'Columns: {list(scores.columns)}')
scores.head()

In [ ]:
# Compute win/loss per round — need to match both sides of each battle
# A round has two perspectives; the one with higher total_score wins that round

# Pair perspectives by battle_id + round
paired = scores.groupby(['battle_id', 'round']).apply(
    lambda g: pd.Series({
        'bot_a': g.iloc[0]['bot'] if len(g) >= 2 else np.nan,
        'bot_b': g.iloc[1]['bot'] if len(g) >= 2 else np.nan,
        'score_a': g.iloc[0]['total_score'] if len(g) >= 2 else np.nan,
        'score_b': g.iloc[1]['total_score'] if len(g) >= 2 else np.nan,
    })
).dropna().reset_index()

paired['winner'] = np.where(paired['score_a'] > paired['score_b'], 
                            paired['bot_a'], paired['bot_b'])
paired['margin'] = abs(paired['score_a'] - paired['score_b'])

print(f'Paired rounds: {len(paired):,}')
print(f'\nTop winners:')
print(paired['winner'].value_counts().head(10))

In [ ]:
# Win rate matrix: for each bot pair, what fraction does bot_a win?
win_rates = []
for (a, b), group in paired.groupby(['bot_a', 'bot_b']):
    wins_a = (group['winner'] == a).sum()
    total = len(group)
    win_rates.append({'bot_a': a, 'bot_b': b, 'win_rate_a': wins_a / total, 'n_rounds': total})

wr_df = pd.DataFrame(win_rates)

# Overall win rate per bot
overall = []
for bot in scores['bot'].unique():
    as_a = wr_df[wr_df['bot_a'] == bot]
    as_b = wr_df[wr_df['bot_b'] == bot]
    total_wins = as_a['win_rate_a'].multiply(as_a['n_rounds']).sum() + \
                 (1 - as_b['win_rate_a']).multiply(as_b['n_rounds']).sum()
    total_rounds = as_a['n_rounds'].sum() + as_b['n_rounds'].sum()
    if total_rounds > 0:
        overall.append({'bot': bot, 'win_rate': total_wins / total_rounds, 'rounds': total_rounds})

overall_df = pd.DataFrame(overall).sort_values('win_rate', ascending=False)
print('Overall win rates (top 15):')
print(overall_df.head(15).to_string(index=False))

## 2. Fire Power Prediction (Question 5)

When the opponent fires, what power do they choose? The initial intuition showed we can detect
WHEN they fire (gun heat). Now: can we predict the POWER they'll use?

This matters for wave surfing — the bullet power determines bullet speed, which determines
the wave's travel time and the maximum escape angle.

In [ ]:
# Load sampled ticks for fire power analysis
np.random.seed(42)
bot_files = {}
for f in all_ticks_files:
    bot_name = f.parent.name
    if bot_name not in bot_files:
        bot_files[bot_name] = []
    bot_files[bot_name].append(f)

frames = []
for bot_name, files in bot_files.items():
    n = min(3, len(files))
    chosen = np.random.choice(len(files), n, replace=False)
    for i in chosen:
        fpath = files[i]
        battle_dir = fpath.parent.parent
        perspectives = [d.name for d in battle_dir.iterdir() if d.is_dir()]
        opponent_name = [p for p in perspectives if p != bot_name]
        if not opponent_name:
            continue
        df = pd.read_csv(fpath)
        df = df[df['scan_available'] == 1].copy()
        df['observer_bot'] = bot_name
        df['opponent_bot'] = opponent_name[0]
        frames.append(df)

ticks = pd.concat(frames, ignore_index=True)

# Filter to firing ticks only
fire_ticks = ticks[ticks['opponent_fired'] == 1].copy()
print(f'Total fire events: {len(fire_ticks):,}')
print(f'Bots that fired: {fire_ticks["opponent_bot"].nunique()}')
print(f'\nFire power distribution:')
print(fire_ticks['opponent_fire_power'].describe())

In [ ]:
# Fire power distribution per bot
fig, ax = plt.subplots(figsize=(14, 6))
top_firers = fire_ticks['opponent_bot'].value_counts().head(20).index
fire_subset = fire_ticks[fire_ticks['opponent_bot'].isin(top_firers)]
fire_subset.boxplot(column='opponent_fire_power', by='opponent_bot', ax=ax, rot=45)
ax.set_title('Fire Power Distribution by Bot (top 20 by fire count)')
ax.set_xlabel('Bot')
ax.set_ylabel('Fire Power')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# Can we predict fire power from game state?
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

feature_cols = [c for c in ticks.columns if c not in 
                ('battle_id', 'round', 'tick', 'scan_available',
                 'observer_bot', 'opponent_bot', 'opponent_fired', 'opponent_fire_power')]

X = fire_ticks[feature_cols].fillna(0).values
y = fire_ticks['opponent_fire_power'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Baseline: predict mean fire power
mean_power = y_train.mean()
baseline_mae = mean_absolute_error(y_test, np.full_like(y_test, mean_power))

# Random Forest
rf = RandomForestRegressor(n_estimators=50, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)

print(f'Fire Power Prediction ({len(y_test):,} test samples):')
print(f'  Mean baseline: MAE={baseline_mae:.3f}')
print(f'  Random Forest: MAE={rf_mae:.3f}, R²={rf_r2:.3f}')
print(f'  Improvement:   {(1 - rf_mae/baseline_mae)*100:.1f}% lower MAE')

# Feature importance
importances = pd.Series(rf.feature_importances_, index=feature_cols)
top10 = importances.nlargest(10)
print(f'\nTop 10 features for fire power prediction:')
for feat, imp in top10.items():
    print(f'  {feat}: {imp:.4f}')

## 3. Game State Clusters at Scale (Question 10)

The initial 5-bot analysis found 3 game state clusters. With 51 bots,
do we see more fine-grained states?

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Use a sample of scan ticks for clustering
cluster_cols = [
    'distance', 'opponent_velocity', 'opponent_lateral_velocity',
    'opponent_heading_delta', 'opponent_dist_to_wall_min',
    'our_lateral_velocity', 'our_dist_to_wall_min',
    'energy_ratio', 'distance_norm'
]

sample = ticks[cluster_cols].dropna().sample(n=min(100000, len(ticks)), random_state=42)
X_clust = StandardScaler().fit_transform(sample.values)

# Elbow + silhouette
ks = range(2, 10)
inertias = []
sil_scores = []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_clust)
    inertias.append(km.inertia_)
    s = silhouette_score(X_clust, labels, sample_size=10000)
    sil_scores.append(s)
    print(f'K={k}: inertia={km.inertia_:.0f}, silhouette={s:.3f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(list(ks), inertias, 'o-')
ax1.set_xlabel('K'); ax1.set_ylabel('Inertia'); ax1.set_title('Elbow Plot')
ax2.plot(list(ks), sil_scores, 'o-')
ax2.set_xlabel('K'); ax2.set_ylabel('Silhouette Score'); ax2.set_title('Silhouette Score')
plt.tight_layout()
plt.show()

In [ ]:
# PCA visualization with best K
best_k = max(zip(ks, sil_scores), key=lambda x: x[1])[0]
print(f'Best K by silhouette: {best_k}')

km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = km_final.fit_predict(X_clust)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_clust)

fig, ax = plt.subplots(figsize=(10, 8))
for c in range(best_k):
    mask = cluster_labels == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], label=f'Cluster {c}', alpha=0.3, s=5)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title(f'Game State Clusters (K={best_k}, {len(X_clust):,} samples)')
ax.legend()
plt.tight_layout()
plt.show()

# Cluster profiles
sample_with_labels = sample.copy()
sample_with_labels['cluster'] = cluster_labels
print('\nCluster profiles (means):')
print(sample_with_labels.groupby('cluster')[cluster_cols].mean().round(2).to_string())

## 4. Winning Game States (Question 11)

For each cluster, what's the average energy_ratio (proxy for who's winning)?
Are some game states associated with better outcomes for certain bot types?

In [ ]:
# Add cluster labels to the full sampled ticks
ticks_for_cluster = ticks[cluster_cols].dropna()
if len(ticks_for_cluster) > 0:
    X_all = StandardScaler().fit_transform(ticks_for_cluster.values)
    # Use the same KMeans model (need to refit on full feature scale)
    scaler = StandardScaler().fit(ticks[cluster_cols].dropna().values)
    X_all_scaled = scaler.transform(ticks_for_cluster.values)
    ticks_clustered = ticks.loc[ticks_for_cluster.index].copy()
    ticks_clustered['cluster'] = km_final.predict(X_all_scaled)
    
    # Energy ratio by cluster and bot type
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Overall energy ratio per cluster
    ticks_clustered.boxplot(column='energy_ratio', by='cluster', ax=axes[0])
    axes[0].set_title('Energy Ratio by Game State Cluster')
    axes[0].axhline(y=0.5, color='red', linestyle='--', alpha=0.5)
    axes[0].set_xlabel('Cluster')
    
    # Cluster distribution per top bot (as observer)
    top5 = ticks_clustered['observer_bot'].value_counts().head(5).index
    cluster_dist = ticks_clustered[ticks_clustered['observer_bot'].isin(top5)].groupby(
        ['observer_bot', 'cluster']).size().unstack(fill_value=0)
    cluster_dist_norm = cluster_dist.div(cluster_dist.sum(axis=1), axis=0)
    cluster_dist_norm.plot(kind='bar', stacked=True, ax=axes[1])
    axes[1].set_title('Game State Distribution by Bot')
    axes[1].set_xlabel('Bot')
    axes[1].set_ylabel('Fraction of Ticks')
    axes[1].legend(title='Cluster', bbox_to_anchor=(1.05, 1))
    
    plt.suptitle('')
    plt.tight_layout()
    plt.show()

## 5. Round Outcome Prediction (Question 6)

With ~25K+ rounds (vs. 100 in the initial analysis), we can build a much better
round-outcome model. Features: aggregated tick-level stats per round.

In [ ]:
# Build per-round features from tick data
round_features = ticks.groupby(['battle_id', 'round', 'observer_bot']).agg(
    distance_mean=('distance', 'mean'),
    distance_std=('distance', 'std'),
    energy_ratio_mean=('energy_ratio', 'mean'),
    opp_lat_vel_std=('opponent_lateral_velocity', 'std'),
    opp_vel_abs_mean=('opponent_velocity', lambda x: x.abs().mean()),
    our_lat_vel_std=('our_lateral_velocity', 'std'),
    opp_fired_count=('opponent_fired', 'sum'),
    opp_heading_delta_std=('opponent_heading_delta', 'std'),
    wall_dist_mean=('opponent_dist_to_wall_min', 'mean'),
    n_ticks=('distance', 'count'),
).reset_index()

# Merge with scores to get round outcome
# scores has: battle_id, round, bot (=observer), total_score, etc.
round_data = round_features.merge(
    scores[['battle_id', 'round', 'bot', 'total_score']],
    left_on=['battle_id', 'round', 'observer_bot'],
    right_on=['battle_id', 'round', 'bot'],
    how='inner'
)

# Label: did the observer win this round? (score > median score for this round)
round_medians = round_data.groupby(['battle_id', 'round'])['total_score'].median()
round_data = round_data.merge(round_medians.rename('median_score'), 
                              on=['battle_id', 'round'])
round_data['won'] = (round_data['total_score'] > round_data['median_score']).astype(int)

print(f'Round-level samples: {len(round_data):,}')
print(f'Win rate: {round_data["won"].mean():.3f}')

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

round_feature_cols = ['distance_mean', 'distance_std', 'energy_ratio_mean',
                      'opp_lat_vel_std', 'opp_vel_abs_mean', 'our_lat_vel_std',
                      'opp_fired_count', 'opp_heading_delta_std', 'wall_dist_mean', 'n_ticks']

X = round_data[round_feature_cols].fillna(0).values
y = round_data['won'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_round = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_round.fit(X_train, y_train)

train_acc = accuracy_score(y_train, rf_round.predict(X_train))
test_acc = accuracy_score(y_test, rf_round.predict(X_test))

print(f'Round Outcome Prediction ({len(y_test):,} test rounds):')
print(f'  Majority baseline: {max(y.mean(), 1-y.mean()):.3f}')
print(f'  Train accuracy:    {train_acc:.3f}')
print(f'  Test accuracy:     {test_acc:.3f}')

# Feature importance
importances = pd.Series(rf_round.feature_importances_, index=round_feature_cols)
print(f'\nFeature importance for round outcome:')
for feat, imp in importances.sort_values(ascending=False).items():
    print(f'  {feat}: {imp:.4f}')

## 6. Summary

**Fire Power Prediction:**
- (filled in after running)

**Game State Clusters:**
- (filled in after running)

**Round Outcomes:**
- (filled in after running)

**Win Rankings:**
- (filled in after running)